# RSNA 2025 — 1st Place Pretrained Inference Notebook

**Input**: A DICOM folder for a single series  
**Output**: 14 probabilities — 13 location scores + 1 aneurysm-present score

## Pipeline (exact winners pipeline)
1. **DICOM → NIfTI** via `dcm2niix` (majority-size selection, slice-spacing filter)
2. **Vessel segmentation** via `VesselSegmentationPredictor` (sparse search → dense nnU-Net)
3. **ROI crop** extracted from segmentation output, z-score normalised
4. **ROI classification** via `AneurysmVesselSegROILitModuleTransformer` (fold ensemble)
5. **Output**: `polars.DataFrame` with 14 columns matching ANEURYSM_CLASSES order

### Key environment variables (match Kaggle submission exactly)
| Variable | Value |
|---|---|
| `VESSEL_DEVICE` | `cuda:0` (or `cpu`) |
| `VESSEL_NNUNET_SPARSE_MODEL_DIR` | `nnunet-vessel-grouping-da7` |
| `VESSEL_NNUNET_MODEL_DIR` | `nnunet-da3-sklr-ep800` |
| `VESSEL_ADDITIONAL_DENSE_MODEL_DIRS` | `nnunet-da6-sklr-w3-tv07` |
| `VESSEL_FOLDS` | `all` |
| `ROI_EXPERIMENTS` | `251013-seg_tf-v4-nnunet_truncate1_preV6_1-ex_dav6w3-m32g64-e25-w01_005_1-s128_256_256` |
| `ROI_FOLDS` | `0,1,2,3` |
| `ROI_TTA` | `2` |

## Cell 1 — Configure paths (edit these)

In [10]:
import os
from pathlib import Path

# ============================================================
# USER CONFIGURATION — adjust before running
# ============================================================

# Path to the DICOM folder you want to run inference on
DICOM_FOLDER = r"C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\sample_data\1.2.826.0.1.3680043.8.498.10022796280698534221758473208024838831"

# Root of the repository (directory that contains pipeline.py, scripts/, src/, configs/, etc.)
REPO_ROOT = Path(r"C:\Users\maila\Desktop\RSNA_Major\rsna2025_main")

# Device: "cuda:0" for GPU inference, "cpu" for CPU-only
DEVICE = "cuda:0"

# ============================================================
# Model directories  (all relative to REPO_ROOT by default)
# These match the Kaggle data-source names exactly
# ============================================================
VESSEL_SPARSE_MODEL_DIR   = str(REPO_ROOT / "nnunet-vessel-grouping-da7")
VESSEL_PRIMARY_MODEL_DIR  = str(REPO_ROOT / "nnunet-da3-sklr-ep800")
VESSEL_ADDL_MODEL_DIRS    = str(REPO_ROOT / "nnunet-da6-sklr-w3-tv07")

# ROI classifier checkpoint directory
# The winner stored checkpoints under a folder named after the experiment
ROI_EXPERIMENT = "251013-seg_tf-v4-nnunet_truncate1_preV6_1-ex_dav6w3-m32g64-e25-w01_005_1-s128_256_256"
ROI_FOLDS      = "0,1,2,3"   # folds 0-3 match Kaggle submission
ROI_TTA        = "2"           # test-time augmentation count (x-flip only)

# Config directory
CONFIG_DIR = str(REPO_ROOT / "configs")

# Debug output (set to "1" to see per-step timing and slice visualisations)
RSNA_DEBUG = "1"

print(f"REPO_ROOT          : {REPO_ROOT}")
print(f"DICOM_FOLDER       : {DICOM_FOLDER}")
print(f"DEVICE             : {DEVICE}")
print(f"Sparse model dir   : {VESSEL_SPARSE_MODEL_DIR}")
print(f"Primary model dir  : {VESSEL_PRIMARY_MODEL_DIR}")
print(f"Additional model   : {VESSEL_ADDL_MODEL_DIRS}")
print(f"ROI experiment     : {ROI_EXPERIMENT}")
print(f"ROI folds          : {ROI_FOLDS}")
print(f"ROI TTA            : {ROI_TTA}")
print(f"Config dir         : {CONFIG_DIR}")

REPO_ROOT          : C:\Users\maila\Desktop\RSNA_Major\rsna2025_main
DICOM_FOLDER       : C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\sample_data\1.2.826.0.1.3680043.8.498.10022796280698534221758473208024838831
DEVICE             : cuda:0
Sparse model dir   : C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\nnunet-vessel-grouping-da7
Primary model dir  : C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\nnunet-da3-sklr-ep800
Additional model   : C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\nnunet-da6-sklr-w3-tv07
ROI experiment     : 251013-seg_tf-v4-nnunet_truncate1_preV6_1-ex_dav6w3-m32g64-e25-w01_005_1-s128_256_256
ROI folds          : 0,1,2,3
ROI TTA            : 2
Config dir         : C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\configs


## Cell 2 — Bootstrap sys.path and PROJECT_ROOT (replaces Kaggle environment setup)

In [2]:
import sys

# Ensure repo root and the bundled nnUNet fork are importable
_paths_to_add = [
    str(REPO_ROOT),
    str(REPO_ROOT / "nnUNet"),
]
for _p in reversed(_paths_to_add):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Set PROJECT_ROOT so Hydra / configs can resolve ${oc.env:PROJECT_ROOT}
os.environ["PROJECT_ROOT"] = str(REPO_ROOT)

# Change working directory to repo root (rootutils / Hydra expect this)
os.chdir(REPO_ROOT)

print("sys.path[0]   :", sys.path[0])
print("sys.path[1]   :", sys.path[1])
print("PROJECT_ROOT  :", os.environ["PROJECT_ROOT"])
print("cwd           :", os.getcwd())

sys.path[0]   : C:\Users\maila\Desktop\RSNA_Major\rsna2025_main
sys.path[1]   : C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\nnUNet
PROJECT_ROOT  : C:\Users\maila\Desktop\RSNA_Major\rsna2025_main
cwd           : C:\Users\maila\Desktop\RSNA_Major\rsna2025_main


## Cell 3 — Set environment variables (exact Kaggle submission env)

In [3]:
# ── Fix ROI checkpoint paths (workaround) ──────────────────────────────────
# The checkpoint loading code looks for checkpoints/ but they're in checkpoint/
# We'll monkey-patch the _find_ckpt_path method after imports to fix this
import sys
_original_find_ckpt_path_patch_applied = False

def _apply_checkpoint_path_fix():
    """Monkey-patch checkpoint path resolution to handle checkpoint vs checkpoints directory"""
    global _original_find_ckpt_path_patch_applied
    if _original_find_ckpt_path_patch_applied:
        return
    
    # Patch will be applied after import - see below
    _original_find_ckpt_path_patch_applied = True

# ── nnUNet paths ───────────────────────────────────────────────────────────
os.environ["nnUNet_raw"]           = str(REPO_ROOT / "logs" / "nnUNet_raw")
os.environ["nnUNet_preprocessed"]  = str(REPO_ROOT / "logs" / "nnUNet_preprocessed")
os.environ["nnUNet_results"]       = str(REPO_ROOT / "logs" / "nnUNet_results")

# ── Vessel segmentation ────────────────────────────────────────────────────
os.environ["VESSEL_DEVICE"]                      = DEVICE
os.environ["VESSEL_DEVICES"]                     = DEVICE.replace("cuda:", "")

# Abort flags (matching Kaggle submission verbatim)
os.environ["VESSEL_ABORT_ON_SPARSE_FAIL"]        = "1"
os.environ["VESSEL_ABORT_MIN_ALL_DIMS_MM"]       = "140"
os.environ["VESSEL_ABORT_ON_SMALL_ROI"]          = "1"
os.environ["VESSEL_MIN_ROI_VOXELS"]              = "1000000"
os.environ["VESSEL_ABORT_ON_LOW_UNION"]          = "1"
os.environ["VESSEL_MIN_UNION_SUM"]               = "5000"

# Fallback probabilities on error (14 values, matching Kaggle submission)
os.environ["RSNA_ERROR_FALLBACK_PROBS"]          = "0.02,0.02,0.08,0.08,0.03,0.03,0.07,0.02,0.02,0.02,0.02,0.02,0.02,0.35"

# Model directories
os.environ["VESSEL_NNUNET_SPARSE_MODEL_DIR"]     = VESSEL_SPARSE_MODEL_DIR
os.environ["VESSEL_NNUNET_MODEL_DIR"]            = VESSEL_PRIMARY_MODEL_DIR
os.environ["VESSEL_ADDITIONAL_DENSE_MODEL_DIRS"] = VESSEL_ADDL_MODEL_DIRS
os.environ["VESSEL_FOLDS"]                       = "all"

# Orientation correction (enabled in Kaggle submission)
os.environ["VESSEL_ENABLE_ORIENTATION_CORRECTION"] = "1"
os.environ["VESSEL_ORIENTATION_WEIGHTS"]           = "1,1,1"

# ROI physical extent
os.environ["VESSEL_SPARSE_ROI_EXTENT_MM"]        = "140"
os.environ["VESSEL_REFINE_Z_ONLY"]               = "0"

# ROI margin refinement
os.environ["VESSEL_REFINE_MARGIN_Z"]             = "15"
os.environ["VESSEL_REFINE_MARGIN_XY"]            = "30"

# Sparse / dense overlap
os.environ["VESSEL_SPARSE_OVERLAP"]              = "0.2"
os.environ["VESSEL_DENSE_OVERLAP"]               = "0.3"

# ── ROI classifier ─────────────────────────────────────────────────────────
os.environ["ROI_EXPERIMENTS"]                    = ROI_EXPERIMENT
os.environ["ROI_FOLDS"]                          = ROI_FOLDS
os.environ["ROI_TTA"]                            = ROI_TTA

# ROI nnUNet model directory override (fixes hardcoded /workspace/ paths in configs)
# This overrides the nnunet_model_dir in AneurysmRoiBackboneNnUNetTruncatedDecoder
os.environ["ROI_NNUNET_MODEL_DIR"]               = VESSEL_PRIMARY_MODEL_DIR

# Config directory (required by load_experiment_config)
os.environ["CONFIG_DIR"]                         = CONFIG_DIR

# Run mode: "kaggle" enables path overrides for ROI nnUNet backbone
os.environ["RUN_MODE"]                           = "kaggle"

# Additional score switches
os.environ["RSNA_ONLY_AP"]                       = "0"
os.environ["RSNA_ONLY_LOCATIONS"]                = "0"

# Error tracking
os.environ["RSNA_MAX_PREDICT_ERRORS"]            = "300"

# Debug / timing
os.environ["RSNA_DEBUG"]                         = RSNA_DEBUG

# Error log: write to a local file
os.environ["RSNA_ERROR_LOG"]                     = str(REPO_ROOT / "predict_errors.jsonl")

print("Environment variables set.")
print(f"nnUNet_raw        : {os.environ['nnUNet_raw']}")
print(f"nnUNet_preprocessed : {os.environ['nnUNet_preprocessed']}")
print(f"nnUNet_results    : {os.environ['nnUNet_results']}")
print(f"ROI_NNUNET_MODEL_DIR: {os.environ['ROI_NNUNET_MODEL_DIR']}")
print(f"RUN_MODE          : {os.environ['RUN_MODE']}")


Environment variables set.
nnUNet_raw        : C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\logs\nnUNet_raw
nnUNet_preprocessed : C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\logs\nnUNet_preprocessed
nnUNet_results    : C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\logs\nnUNet_results
ROI_NNUNET_MODEL_DIR: C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\nnunet-da3-sklr-ep800
RUN_MODE          : kaggle


## Cell 4 — Import the predict function from the script

In [4]:
from scripts.rsna_submission_roi import predict

print("predict() imported successfully from scripts.rsna_submission_roi")

predict() imported successfully from scripts.rsna_submission_roi


In [5]:
# Monkey-patch checkpoint loading to locate experiments in repo root
# Actual experiments are in {REPO_ROOT}/251013-*/{experiment_name} but code looks in {REPO_ROOT}/logs/train/runs/

import re
from pathlib import Path
from scripts.rsna_submission_roi import RsnaRoiPipeline

# Save original method
_original_find_ckpt_path = RsnaRoiPipeline._find_ckpt_path

def _find_experiment_dir(experiment_name: str, repo_root: Path) -> Path | None:
    """Search for experiment directory in repo root (named 251013-* pattern)"""
    if not repo_root.exists():
        return None
    
    # Look for directories matching 251013-* pattern
    for candidate_dir in repo_root.glob("251013-*"):
        if not candidate_dir.is_dir():
            continue
        
        # Check if the experiment directory exists within
        exp_path = candidate_dir / experiment_name
        if exp_path.is_dir():
            return exp_path
    
    return None

def _patched_find_ckpt_path(self, log_dir: Path):
    """Patched version: tries multiple locations to find checkpoints"""
    mode = (self.opts.roi_ckpt or "last").strip().lower()
    
    # Try original location first
    if mode == "best":
        epoch_ckpts = list(log_dir.glob("epoch_*.ckpt"))
    else:
        epoch_ckpts = []
    
    # If original search path doesn't have checkpoints, search in repo root
    search_dirs = [log_dir]
    
    # Also try fallback: checkpoint/ dir instead of checkpoints/
    if (log_dir.parent / "checkpoint").exists():
        search_dirs.append(log_dir.parent / "checkpoint")
    
    # Try to find experiment in repo root
    try:
        if hasattr(self, 'opts') and hasattr(self.opts, 'roi_experiments'):
            exp_name = self.opts.roi_experiments[0] if self.opts.roi_experiments else None
            if exp_name:
                exp_dir = _find_experiment_dir(exp_name, REPO_ROOT)
                if exp_dir:
                    # Get fold number from log_dir path
                    fold_match = re.search(r'fold(\d+)', log_dir.name)
                    if fold_match:
                        fold_num = fold_match.group(1)
                        # Try checkpoint/ subdir in found experiment
                        exp_ckpt_dir = exp_dir / "checkpoint" / f"fold{fold_num}"
                        if exp_ckpt_dir.exists():
                            search_dirs.append(exp_ckpt_dir)
    except Exception:
        pass
    
    if mode == "best":
        for search_dir in search_dirs:
            epoch_ckpts = list(search_dir.glob("epoch_*.ckpt"))
            if epoch_ckpts:
                def _epoch_num(p: Path) -> int:
                    m = re.search(r"epoch[_=]?(\d+)", p.name)
                    if m:
                        try:
                            return int(m.group(1))
                        except Exception:
                            return -1
                    return -1
                return max(epoch_ckpts, key=_epoch_num)
        return None
    
    # Default: last.ckpt
    for search_dir in search_dirs:
        last_ckpt = search_dir / "last.ckpt"
        if last_ckpt.exists():
            if self.opts.debug:
                import logging
                logging.getLogger(__name__).debug(f"[DEBUG] Found checkpoint: {last_ckpt}")
            return last_ckpt
    
    return None

# Apply the patch
RsnaRoiPipeline._find_ckpt_path = _patched_find_ckpt_path
print("Applied enhanced checkpoint path patch to search repo root for experiments")


Applied enhanced checkpoint path patch to search repo root for experiments


## Cell 5 — Warmup run load models into memory

In [ ]:

""" WARMUP_PATH = r"C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\sample_data\1.2.826.0.1.3680043.8.498.10004684224894397679901841656954650085"  # or a separate lightweight sample

print(f"Running warmup on: {WARMUP_PATH}")
warmup_result = predict(WARMUP_PATH)
print("Warmup complete. Result shape:", warmup_result.shape)
print(warmup_result)""" 

DEBUG: [DEBUG] series_path=C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\sample_data\1.2.826.0.1.3680043.8.498.10004684224894397679901841656954650085
DEBUG: [DEBUG] device=cuda:0


Running warmup on: C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\sample_data\1.2.826.0.1.3680043.8.498.10004684224894397679901841656954650085


DEBUG: [DEBUG][STEP1] DICOM→NIfTI conversion logs:
DEBUG:   Audit C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\sample_data\1.2.826.0.1.3680043.8.498.10004684224894397679901841656954650085: dicom=147, majority=(512, 512, 0.45, 0.45), series=1
DEBUG:   Prepared majority subset: 147 files -> C:\Users\maila\AppData\Local\Temp\tmpmm0ztsga (with slice spacing filter)
DEBUG:   dcm2niix direct failed; trying gdcmconv --raw -> dcm2niix
DEBUG:   gdcmconv processed 147 files
DEBUG: [DEBUG][STEP1 DICOM→NIfTI conversion] 27.497s
DEBUG: [DEBUG][STEP2 NIfTI load] 0.293s


Available folds: ['all']
Initializing adaptive sparse-search Predictor...
GPUs: 0
Main device: cuda:0
モデルを読み込み中 (fold: all)...
モデル読み込み完了: 単一fold
モデルを読み込み中 (fold: all)...
モデル読み込み完了: 単一fold
モデルを読み込み中 (fold: all)...
モデル読み込み完了: 単一fold
Models loaded: 25.13s, folds: ('all',)
Sparse model: C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\nnunet-vessel-grouping-da7 (fold=all)
Dense model [primary]: C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\nnunet-da3-sklr-ep800 (device=cuda:0)
Additional dense model [extra_0]: C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\nnunet-da6-sklr-w3-tv07 (device=cuda:0)
疎探索(DBSCAN)フェーズを開始...
ダウンサンプリング: torch.Size([1, 103, 229, 189]) -> torch.Size([1, 103, 229, 189])
Input shape: torch.Size([1, 103, 229, 189])
step_size: 0.8
mirror_axes: None
n_steps 4, image size is torch.Size([128, 229, 189]), tile_size [128, 128, 128], tile_step_size 0.8
steps:
[[0], [0, 101], [0, 61]]
move image to device cuda:0
preallocating results arrays on device cuda:0
running prediction

DEBUG: [DEBUG][STEP3 VesselSeg inference + ROI extraction] 48.349s
DEBUG: [DEBUG] roi_voxels=5084744
DEBUG: [DEBUG] Orientation correction: perm=[0, 1, 2] signs=[1, 1, 1] score=4.934
DEBUG: Setting JobRuntime:name=UNKNOWN_NAME
DEBUG: Setting JobRuntime:name=app
DEBUG: [DEBUG] ROI nnUNet dir (exp=251013-seg_tf-v4-nnunet_truncate1_preV6_1-ex_dav6w3-m32g64-e25-w01_005_1-s128_256_256): C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\nnunet-da3-sklr-ep800
c:\Users\maila\AppData\Local\Programs\Python\Python310\lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'net' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['net'])`.
c:\Users\maila\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
DEBUG: 

Warmup complete. Result shape: (1, 14)
shape: (1, 14)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Left Infr ┆ Right Inf ┆ Left Supr ┆ Right Sup ┆ … ┆ Right     ┆ Basilar   ┆ Other     ┆ Aneurysm │
│ aclinoid  ┆ raclinoid ┆ aclinoid  ┆ raclinoid ┆   ┆ Posterior ┆ Tip       ┆ Posterior ┆ Present  │
│ Internal  ┆ Internal  ┆ Internal  ┆ Internal  ┆   ┆ Communica ┆ ---       ┆ Circulati ┆ ---      │
│ Car…      ┆ Ca…       ┆ Car…      ┆ Ca…       ┆   ┆ ting …    ┆ f64       ┆ on        ┆ f64      │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆           ┆ ---       ┆          │
│ f64       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆           ┆ f64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 0.007122  ┆ 0.00227   ┆ 0.003538  ┆ 0.003311  ┆ … ┆ 0.001942  ┆ 0.001525  ┆ 0.002522  ┆ 0.047943 │
└───────────┴───────────┴───────────┴

## Cell 6 — Full inference on your DICOM folder

In [11]:
import polars as pl

# Run the full pipeline
result: pl.DataFrame = predict(DICOM_FOLDER)

print("\n" + "="*70)
print("  INFERENCE RESULT")
print("="*70)
print(result)

DEBUG: [DEBUG] series_path=C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\sample_data\1.2.826.0.1.3680043.8.498.10022796280698534221758473208024838831
DEBUG: [DEBUG] device=cuda:0
DEBUG: [DEBUG][STEP1] DICOM→NIfTI conversion logs:
DEBUG:   Audit C:\Users\maila\Desktop\RSNA_Major\rsna2025_main\sample_data\1.2.826.0.1.3680043.8.498.10022796280698534221758473208024838831: dicom=671, majority=(512, 512, 0.46, 0.46), series=1
DEBUG:   Prepared majority subset: 671 files -> C:\Users\maila\AppData\Local\Temp\tmpa1zqr1qp (with slice spacing filter)
DEBUG:   dcm2niix direct failed; trying gdcmconv --raw -> dcm2niix
DEBUG:   gdcmconv processed 671 files
DEBUG: [DEBUG][STEP1 DICOM→NIfTI conversion] 129.674s
DEBUG: [DEBUG][STEP2 NIfTI load] 0.762s


疎探索(DBSCAN)フェーズを開始...
ダウンサンプリング: torch.Size([1, 336, 236, 236]) -> torch.Size([1, 336, 236, 236])
Input shape: torch.Size([1, 336, 236, 236])
step_size: 0.8
mirror_axes: None
n_steps 36, image size is torch.Size([336, 236, 236]), tile_size [128, 128, 128], tile_step_size 0.8
steps:
[[0, 69, 139, 208], [0, 54, 108], [0, 54, 108]]
move image to device cuda:0
preallocating results arrays on device cuda:0
running prediction: 36 steps
DBSCAN-sparse SIガード: 軸0, 物理長 336.0mm (最大 336.0mm) > 上限 150.0mm
 -> 低解像度で下側 186 voxel を背景化
DBSCAN-sparse SIガード(重心調整): COM=211, 範囲[136:286] vox, 半幅 75vox — 許容窓を元予測に復元
Input shape: torch.Size([1, 176, 315, 317])
step_size: 0.7
mirror_axes: None
n_steps 16, image size is torch.Size([176, 315, 317]), tile_size [64, 192, 192], tile_step_size 0.7
steps:
[[0, 37, 75, 112], [0, 123], [0, 125]]
move image to device cuda:0
preallocating results arrays on device cuda:0
running prediction: 16 steps
Input shape: torch.Size([1, 141, 269, 281])
step_size: 0.7
mirror_axes: Non

DEBUG: [DEBUG][STEP3 VesselSeg inference + ROI extraction] 25.120s
DEBUG: [DEBUG] roi_voxels=10658049
DEBUG: [DEBUG] Orientation correction: perm=[0, 1, 2] signs=[1, 1, 1] score=5.161
DEBUG: [DEBUG] union_b=21999
DEBUG: [DEBUG] ROI TTA variants=2 flags=[(False, False, False), (False, False, True)]
DEBUG: [DEBUG][STEP4 ROI classification inference] 207.329s
DEBUG: [DEBUG][STEP5 Fold ensemble (mean + post)] 0.021s
DEBUG: [DEBUG] probs shape=(14,) min=0.0017 max=0.9937



  INFERENCE RESULT
shape: (1, 14)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Left Infr ┆ Right Inf ┆ Left Supr ┆ Right Sup ┆ … ┆ Right     ┆ Basilar   ┆ Other     ┆ Aneurysm │
│ aclinoid  ┆ raclinoid ┆ aclinoid  ┆ raclinoid ┆   ┆ Posterior ┆ Tip       ┆ Posterior ┆ Present  │
│ Internal  ┆ Internal  ┆ Internal  ┆ Internal  ┆   ┆ Communica ┆ ---       ┆ Circulati ┆ ---      │
│ Car…      ┆ Ca…       ┆ Car…      ┆ Ca…       ┆   ┆ ting …    ┆ f64       ┆ on        ┆ f64      │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆           ┆ ---       ┆          │
│ f64       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆           ┆ f64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 0.0042    ┆ 0.004005  ┆ 0.003248  ┆ 0.00428   ┆ … ┆ 0.004349  ┆ 0.001972  ┆ 0.004135  ┆ 0.993652 │
└───────────┴───────────┴───────────┴───────────┴───┴───

## Cell 7 — Display results with labels

In [12]:
import numpy as np
from src.data.components.aneurysm_vessel_seg_dataset import ANEURYSM_CLASSES

# ANEURYSM_CLASSES = 13 location labels + "Aneurysm Present" (last)
probs = result.to_numpy().flatten()

print(f"{'Label':<55} {'Probability':>12}")
print("-" * 70)
for label, prob in zip(ANEURYSM_CLASSES, probs):
    bar = "█" * int(prob * 30)
    flag = "  ← POSITIVE" if prob > 0.5 else ""
    print(f"{label:<55} {prob:>10.4f}  {bar}{flag}")

print()
print(f"Aneurysm Present probability : {probs[-1]:.4f}")
print(f"Predicted                    : {'POSITIVE' if probs[-1] > 0.5 else 'NEGATIVE'}")

Label                                                    Probability
----------------------------------------------------------------------
Left Infraclinoid Internal Carotid Artery                   0.0042  
Right Infraclinoid Internal Carotid Artery                  0.0040  
Left Supraclinoid Internal Carotid Artery                   0.0032  
Right Supraclinoid Internal Carotid Artery                  0.0043  
Left Middle Cerebral Artery                                 0.0021  
Right Middle Cerebral Artery                                0.9751  █████████████████████████████  ← POSITIVE
Anterior Communicating Artery                               0.0018  
Left Anterior Cerebral Artery                               0.0017  
Right Anterior Cerebral Artery                              0.0019  
Left Posterior Communicating Artery                         0.0047  
Right Posterior Communicating Artery                        0.0043  
Basilar Tip                                                 

---
## Notes

### What `predict(series_path)` does internally

1. **DICOM → NIfTI** — calls `convert_dicom_to_nifti()` from `src.my_utils.rsna_dcm2niix`  
   Flags: `-z n -b y -i n -f %s`, majority-size filter, slice-spacing tolerance 2.0
   
2. **NIfTI load** — `SimpleITKIOWithReorient.read_images([path], orientation="RAS")`

3. **Vessel segmentation** — `VesselSegmentationPredictor.predict_single_volume_with_info()`  
   - Sparse search model: `nnunet-vessel-grouping-da7` (4 classes, orientation correction)  
   - Dense model 1: `nnunet-da3-sklr-ep800` (folds=all)  
   - Dense model 2: `nnunet-da6-sklr-w3-tv07` (additional dense, recall-focused)  
   - ROI refine margins: Z=15 voxels, XY=30 voxels

4. **ROI classification** — `AneurysmVesselSegROILitModuleTransformer`  
   - Experiment: `251013-seg_tf-v4-nnunet_truncate1_preV6_1-ex_dav6w3-m32g64-e25-w01_005_1-s128_256_256`  
   - Folds: 0, 1, 2, 3 (ensemble mean)  
   - TTA: 2 (original + x-flip)  
   - Input size: 128×256×256

5. **Output** — `polars.DataFrame` with 14 columns (sigmoid probabilities)

### Required external tools
- `dcm2niix` — install from https://github.com/rordenlab/dcm2niix/releases  
- `gdcmconv` — part of the GDCM toolkit (install `libgdcm-tools` on Linux or via conda)

### GPU Memory
The pipeline loads three nnU-Net models plus the ROI classifier.  
Minimum recommended: **16 GB VRAM** (or reduce to CPU with `DEVICE = "cpu"`).